# XTTS Fine-tune WebUI — Colab (2026 guncel)

Bu notebook `xtts-finetune-webui` reposunu **Colab'in guncel runtime'i**
(Python 3.12 + torch 2.8) ile calisacak sekilde kurar.

**Eski notebooktan farklari:**
- torch **downgrade edilmiyor** (eski notebook torch 2.1.2 kurmaya calisiyordu, Py3.12'de o wheel yok)
- `coqui-tts` 0.24 -> 0.27+ (idiap fork, aktif bakiliyor)
- `fastapi` / `pydantic` pinleri kaldirildi (gradio ile cakisiyorlardi)
- Olu `coqui.gateway.scarf.sh` linkleri HuggingFace'e tasindi
- `torch.load(..., weights_only=False)` (torch 2.6+ icin zorunlu)

> **Runtime -> Change runtime type -> GPU (L4 veya A100)** secmeyi unutmayin.

In [ ]:
#@title 1. Ortam kontrolu
import sys, subprocess
print("Python :", sys.version.split()[0])
import torch
print("torch  :", torch.__version__, "| CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
else:
    print("!!! GPU YOK - Runtime > Change runtime type > GPU secin")

In [ ]:
#@title 2. Bagimliliklar
# ONEMLI: torch'a DOKUNMUYORUZ. Colab'in hazir torch'u (>=2.2) zaten yeterli.
!apt-get -qq install -y libegl1 libopengl0 libxcb-cursor0 ffmpeg > /dev/null

!pip install -q --upgrade "coqui-tts[languages]>=0.27.0" "faster_whisper>=1.1.0" "gradio>=5.0.0,<6.0.0" "spacy>=3.8.0" "cutlet>=0.5.0" "fugashi[unidic-lite]>=1.4.0"

# Dogrulama - burada patlarsa devam etmenin anlami yok
import importlib
for mod, label in [("TTS","coqui-tts"), ("trainer","coqui-tts-trainer"),
                   ("gradio","gradio"), ("faster_whisper","faster-whisper")]:
    m = importlib.import_module(mod)
    print("OK ", label, getattr(m, "__version__", "?"))

from TTS.tts.configs.xtts_config import XttsAudioConfig
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
print("OK  XTTS trainer importlari calisiyor")

In [ ]:
#@title 3. Repoyu klonla
REPO_URL = "https://github.com/daswer123/xtts-finetune-webui.git"  #@param {type:"string"}
BRANCH   = "modernize-2026"  #@param {type:"string"}

%cd /content
!rm -rf xtts-finetune-webui
!git clone -q -b "$BRANCH" "$REPO_URL" xtts-finetune-webui || git clone -q "$REPO_URL" xtts-finetune-webui
%cd /content/xtts-finetune-webui
!git log -1 --oneline

In [ ]:
#@title 4. Egitim ayarlari (VRAM'e gore otomatik)
import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Coqui'nin resmi XTTS tarifi: batch_size * grad_acumm >= 252 olmali.
# Kucuk batch + yuksek accumulation, buyuk batch'ten daha stabil sonuc veriyor.
if vram >= 38:        # A100 40/80GB
    BATCH_SIZE, GRAD_ACUMM = 12, 21
elif vram >= 22:      # L4 24GB
    BATCH_SIZE, GRAD_ACUMM = 6, 42
else:                 # T4 16GB
    BATCH_SIZE, GRAD_ACUMM = 3, 84

NUM_EPOCHS       = 10   #@param {type:"integer"}
MAX_AUDIO_LENGTH = 11   #@param {type:"integer"}
WHISPER_MODEL    = "large-v3"  #@param ["large-v3","large-v2","medium","small"]

print("VRAM %.0fGB -> batch_size=%d, grad_acumm=%d (efektif batch %d)"
      % (vram, BATCH_SIZE, GRAD_ACUMM, BATCH_SIZE*GRAD_ACUMM))

In [ ]:
#@title 5. Arayuzu baslat
# Ciktida bir gradio.live linki cikacak, ona tiklayin.
# Adimlar: 1) Data processing  2) Fine-tuning  2.5) Optimize  3) Inference testi
%cd /content/xtts-finetune-webui
!python xtts_demo.py --share \
    --batch_size $BATCH_SIZE \
    --grad_acumm $GRAD_ACUMM \
    --num_epochs $NUM_EPOCHS \
    --max_audio_length $MAX_AUDIO_LENGTH \
    --whisper_model $WHISPER_MODEL

## Sonuclari kaydetme

Colab oturumu kapaninca her sey silinir. Egitim bitince **mutlaka** asagidaki
hucrelerden birini calistirin.

In [ ]:
#@title 6a. Google Drive'a kaydet (onerilen)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/xtts-finetune-webui/finetune_models/ready'
dst = '/content/drive/MyDrive/xtts_finetune_ready'
assert os.path.isdir(src), "ready klasoru yok - egitim/optimize adimi tamamlanmamis"
shutil.copytree(src, dst, dirs_exist_ok=True)
print("Kaydedildi:", dst)
print(os.listdir(dst))

In [ ]:
#@title 6b. Zip olarak indir
import shutil, os
src = '/content/xtts-finetune-webui/finetune_models/ready'
assert os.path.isdir(src), "ready klasoru yok - egitim/optimize adimi tamamlanmamis"
shutil.make_archive('/content/xtts_ready', 'zip', src)
print("Boyut: %.1f MB" % (os.path.getsize('/content/xtts_ready.zip')/1024**2))
from google.colab import files
files.download('/content/xtts_ready.zip')